In [ ]:
# KALMAN LEGACY NET DELTA FORENSICS v2.3 — ONE CELL / READ ONLY
# Resolve the <=5.31e-05 difference before any canonical write.
from google.colab import drive
drive.mount("/content/drive",force_remount=False)
from pathlib import Path
import numpy as np,pandas as pd

R=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results")
A=pd.read_parquet(R/"open_revalidation_v1/open_revalidation_trade_audit.parquet")
L=pd.read_parquet(R/"exit_policy_v1_0_pre2026/exit_policy_v1_0_1_trade_ledger.parquet")
L=L[L.policy.eq("FIXED_4")].copy()
keys=["policy","fold","entry_timestamp","exit_timestamp","entry_seq","exit_seq","symbol"]
for c in ["entry_timestamp","exit_timestamp"]:
 A[c]=pd.to_datetime(A[c],utc=True); L[c]=pd.to_datetime(L[c],utc=True)
M=A.merge(L[keys+["gross_return","net_return","weight"]],on=keys,how="left",suffixes=("","__ledger"),validate="one_to_one")
old=M["reconstructed_fixed4_net_return"].notna()
Z=M.loc[old,keys+["weight","gross_return","net_return","reconstructed_fixed4_raw_return","cost_proxy","reconstructed_fixed4_net_return","net_return__ledger"]].copy()
Z["legacy_minus_ledger"]=Z["reconstructed_fixed4_net_return"]-Z["net_return__ledger"]
Z["ledger_cost"]=Z["gross_return"]-Z["net_return__ledger"]
Z["legacy_cost_vs_ledger_gross"]=Z["gross_return"]-Z["reconstructed_fixed4_net_return"]
Z["abs_delta"]=Z["legacy_minus_ledger"].abs()
print("[ROWS]",len(Z))
print("\n[DELTA TABLE]\n",Z.sort_values("abs_delta",ascending=False).to_string(index=False))
print("\n[SUMMARY]\n",Z[["legacy_minus_ledger","ledger_cost","cost_proxy"]].describe().to_string())
print("\nexact_equal=",int(np.isclose(Z["legacy_minus_ledger"],0,atol=1e-12,rtol=0).sum()),"/",len(Z))
print("max_abs_delta=",Z["abs_delta"].max())
print("median_abs_delta=",Z["abs_delta"].median())

# Test whether legacy reconstructed net was based on IEX raw * strategy weight - cost.
raw=pd.to_numeric(Z["reconstructed_fixed4_raw_return"],errors="coerce")
w=pd.to_numeric(Z["weight"],errors="coerce")
cp=pd.to_numeric(Z["cost_proxy"],errors="coerce")
target=pd.to_numeric(Z["reconstructed_fixed4_net_return"],errors="coerce")
forms={
 "iex_raw_times_weight_minus_costproxy":raw*w-cp,
 "iex_raw_times_weight_minus_ledger_cost":raw*w-Z["ledger_cost"],
 "iex_raw_times_weight":raw*w,
 "ledger_net":Z["net_return__ledger"],
}
print("\n[FORMULA ERRORS]")
for name,v in forms.items():
 e=(target-v).abs()
 print(name,"median=",float(e.median()),"max=",float(e.max()))

print("\nREAD ONLY — no files modified.")
print("Decision rule: if IEX_raw*weight-cost_proxy reproduces legacy to numerical tolerance, use that exact contract for all 888 complete rows; otherwise preserve ledger net as baseline and keep IEX-derived net as a separate field rather than overwriting semantics.")
